In [ ]:
import pandas as pd

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.8 MB/s eta 0:00:00


In [ ]:
!pip install biopython numpy pandas scikit-learn matplotlib


In [ ]:
from Bio import SeqIO
import pandas as pd

def parse_fasta_with_label(filepath, label):
    records = list(SeqIO.parse(filepath, "fasta"))
    return [{"id": r.id, "sequence": str(r.seq), "label": label} for r in records]

dsb = parse_fasta_with_label("DSB.fa.txt", 1)
ssb = parse_fasta_with_label("SSB.fa.txt", 1)
non_nabp = parse_fasta_with_label("Non-NABP.fa.txt", 0)

# Combine all data
data = dsb + ssb + non_nabp
df = pd.DataFrame(data)

In [ ]:
df = pd.read_csv('/content/Rna_binding_dataset.csv')
df

,id,sequence,label
0,A0A0H3MDW1,MAGPKHVLLVSEHWDLFFQTKELLNPEEYRCTIGQQYKQELSADLV...,1
1,A0A1D5NS60,MSRKRNHCYMETGASSESQGAFVDSAGPFSRDEEDFSELEPDEQLV...,1
2,A0A291NUG3,MLHSSLHPSIPQPRGRRAKKAAFVLLSVCLVVLWDLGERPEHILQW...,1
3,A0A3Q7HRZ6,MTEYSLPTMNLWNNSTSDDNVSMMEAFMSSDLSFWATNNSTSAAVV...,1
4,A0A5E4M3Q4,MSVCVSPLVQATILMTEESLTCPQCPKSFSSTKLLQQHQQMFHTDK...,1
...,...,...,...
17452,Q9ZD84,MIDKSSADILEISAKCGYLTGLLKNVYRSANIIVTDMSPLLLDSFD...,0
17453,Q9ZDC7,MKEKPIIGVTPDLAKNCQKYTYADFPWYALRRNYTDAIIAAGGIPI...,0
17454,Q9ZDF9,MILNMTINYIKKDRYKVLNLVLITISIIALSTAYIAEYIFHYTPCP...,0
17455,Q9ZDG1,MNILITALEQSLIMLPLILGMYISYRILKITDLTVDGTYVLGAAVF...,0


In [ ]:
# Save to CSV
df.to_csv("Rna_binding_dataset.csv", index=False)

# Let user download the file
from google.colab import files
files.download("Rna_binding_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df['label'].value_counts()

,count
label,
0,12899
1,4558


Now apply
1. AAC

In [ ]:
import pandas as pd
from collections import Counter
import os

# Define standard 20 amino acids
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'

def calculate_aac(sequence):
    length = len(sequence)
    counts = Counter(sequence)
    return [counts[aa] / length if length > 0 else 0 for aa in amino_acids]

# Load your dataset
df = pd.read_csv('/content/Rna_binding_dataset.csv')

# Assumes your sequence column is named 'Sequence'. Change if needed.
if 'sequence' not in df.columns:
    raise ValueError("Expected a column named 'Sequence' in the dataset.")

# Apply AAC to each sequence
aac_features = df['sequence'].apply(calculate_aac)
aac_df = pd.DataFrame(aac_features.tolist(), columns=[f'AAC_{aa}' for aa in amino_acids])

# Combine with original label column if present
if 'label' in df.columns:
    aac_df['label'] = df['label']

# Save the new dataset


aac_df.to_csv("aac_features_dataset.csv", index=False)

# Let user download the file
from google.colab import files
files.download("aac_features_dataset.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# DPC Feature Extraction

In [ ]:
import pandas as pd
from collections import Counter
from itertools import product
from google.colab import files

# Define standard 20 amino acids
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
dipeptides = [aa1 + aa2 for aa1, aa2 in product(amino_acids, repeat=2)]

# DPC calculation function
def calculate_dpc(sequence):
    sequence = sequence.upper()
    total = len(sequence) - 1
    counts = Counter([sequence[i:i+2] for i in range(total)])
    return [counts[dp] / total if total > 0 else 0 for dp in dipeptides]

# Load your dataset
df = pd.read_csv('/content/Rna_binding_dataset.csv')

# Check sequence column
if 'sequence' not in df.columns:
    raise ValueError("Expected a column named 'sequence' in the dataset.")

# Apply DPC
dpc_features = df['sequence'].apply(calculate_dpc)
dpc_df = pd.DataFrame(dpc_features.tolist(), columns=[f'DPC_{dp}' for dp in dipeptides])

# Add label if available
if 'label' in df.columns:
    dpc_df.insert(0, 'label', df['label'])

# Save the new CSV
dpc_df.to_csv('dpc_features_dataset.csv', index=False)

# Let user download the file
files.download('dpc_features_dataset.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# PAAC Feature Extraction

In [ ]:
import pandas as pd
from collections import Counter
import numpy as np
from google.colab import files

# Define 20 standard amino acids and a hydrophobicity scale
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
hydrophobicity = {
    'A': 0.62, 'C': 0.29, 'D': -0.9, 'E': -0.74, 'F': 1.19,
    'G': 0.48, 'H': -0.4, 'I': 1.38, 'K': -1.5, 'L': 1.06,
    'M': 0.64, 'N': -0.78, 'P': 0.12, 'Q': -0.85, 'R': -2.53,
    'S': -0.18, 'T': -0.05, 'V': 1.08, 'W': 0.81, 'Y': 0.26
}

# Parameters
lambda_val = 10  # number of correlation factors
weight = 0.05    # weight of correlation vs. AAC

# Function to calculate PAAC
def calculate_paac(sequence):
    sequence = sequence.upper()
    length = len(sequence)

    # AAC part
    aac = [sequence.count(aa) / length if length > 0 else 0 for aa in amino_acids]

    # Normalize hydrophobicity
    values = [hydrophobicity.get(res, 0) for res in sequence]
    mean_val = np.mean(values)
    std_val = np.std(values) if np.std(values) > 0 else 1
    norm_values = [(v - mean_val) / std_val for v in values]

    # Lambda correlation part
    theta = []
    for l in range(1, lambda_val + 1):
        sum_corr = 0
        for i in range(len(norm_values) - l):
            sum_corr += (norm_values[i] - norm_values[i + l]) ** 2
        theta.append(sum_corr / (length - l) if length > l else 0)

    # Combine
    denominator = 1 + weight * sum(theta)
    paac = [(v / denominator) for v in aac] + [(weight * t / denominator) for t in theta]
    return paac

# Load dataset
df = pd.read_csv('/content/Rna_binding_dataset.csv')

# Check sequence column
if 'sequence' not in df.columns:
    raise ValueError("Expected a column named 'sequence' in the dataset.")

# Apply PAAC
paac_features = df['sequence'].apply(calculate_paac)
aac_headers = [f'PAAC_AAC_{aa}' for aa in amino_acids]
theta_headers = [f'PAAC_Theta_{i}' for i in range(1, lambda_val + 1)]
paac_df = pd.DataFrame(paac_features.tolist(), columns=aac_headers + theta_headers)

# Add label
if 'label' in df.columns:
    paac_df.insert(0, 'label', df['label'])

# Save to CSV
paac_df.to_csv('paac_features_dataset.csv', index=False)
files.download('paac_features_dataset.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# CTDT

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# Hydrophobicity grouping: 3 classes
# Group 1: hydrophobic (A, V, L, I, P, F, M, W)
# Group 2: neutral (G, S, T, Y, C, N, Q)
# Group 3: hydrophilic (D, E, K, R, H)

group_map = {
    'A': 1, 'V': 1, 'L': 1, 'I': 1, 'P': 1, 'F': 1, 'M': 1, 'W': 1,
    'G': 2, 'S': 2, 'T': 2, 'Y': 2, 'C': 2, 'N': 2, 'Q': 2,
    'D': 3, 'E': 3, 'K': 3, 'R': 3, 'H': 3
}

group_names = ['G1', 'G2', 'G3']
transition_pairs = [('G1', 'G2'), ('G1', 'G3'), ('G2', 'G3')]

def calculate_ctdt(sequence):
    sequence = sequence.upper()
    length = len(sequence)

    groups = [group_map.get(aa, 0) for aa in sequence if aa in group_map]
    if not groups or len(groups) < 2:
        return [0]*21  # 3 composition + 3 transition + 15 distribution features

    # Composition
    comp = [groups.count(i)/len(groups) for i in range(1, 4)]

    # Transition
    trans_counts = {pair: 0 for pair in transition_pairs}
    for i in range(len(groups)-1):
        g1, g2 = groups[i], groups[i+1]
        if g1 != g2:
            label = tuple(sorted([f'G{g1}', f'G{g2}']))
            if label in trans_counts:
                trans_counts[label] += 1
    trans = [trans_counts[pair] / (len(groups)-1) for pair in transition_pairs]

    # Distribution
    dist = []
    for i in range(1, 4):
        idx = [j for j, g in enumerate(groups) if g == i]
        if len(idx) == 0:
            dist += [0]*5
        else:
            percentiles = np.percentile(idx, [0, 25, 50, 75, 100])
            dist += [p / (length-1) for p in percentiles]

    return comp + trans + dist

# Load dataset
df = pd.read_csv('/content/Rna_binding_dataset.csv')

# Check sequence column
if 'sequence' not in df.columns:
    raise ValueError("Expected a column named 'sequence' in the dataset.")

# Apply CTDT
ctdt_features = df['sequence'].apply(calculate_ctdt)

headers = (
    [f'CTDT_Comp_{g}' for g in group_names] +
    [f'CTDT_Trans_{g1}_{g2}' for g1, g2 in transition_pairs] +
    [f'CTDT_Dist_{g}_P{i}' for g in group_names for i in [0, 25, 50, 75, 100]]
)
ctdt_df = pd.DataFrame(ctdt_features.tolist(), columns=headers)

# Add label if present
if 'label' in df.columns:
    ctdt_df.insert(0, 'label', df['label'])

# Save to CSV
ctdt_df.to_csv('ctdt_features_dataset.csv', index=False)
files.download('ctdt_features_dataset.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Distance-Pair (DP)

In [ ]:
import pandas as pd
from collections import Counter
from itertools import product
from google.colab import files

# Define standard amino acids and dipeptides
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
pairs = [a1 + a2 for a1, a2 in product(amino_acids, repeat=2)]

# Parameters
max_distance = 5  # you can adjust this

# Function to calculate distance-pair features
def calculate_distance_pair(sequence, max_k=5):
    sequence = sequence.upper()
    features = []

    for k in range(1, max_k + 1):
        counts = Counter()
        total = 0
        for i in range(len(sequence) - k - 1):
            pair = sequence[i] + sequence[i + k + 1]
            if pair in pairs:
                counts[pair] += 1
                total += 1
        features.extend([counts[pair] / total if total > 0 else 0 for pair in pairs])
    return features

# Load dataset
df = pd.read_csv('/content/Rna_binding_dataset.csv')

# Validate input
if 'sequence' not in df.columns:
    raise ValueError("Expected a column named 'sequence' in the dataset.")

# Apply DP Composition
dp_features = df['sequence'].apply(lambda seq: calculate_distance_pair(seq, max_k=max_distance))

# Construct headers
headers = []
for k in range(1, max_distance + 1):
    headers.extend([f'DP_k{k}_{p}' for p in pairs])

# Create DataFrame
dp_df = pd.DataFrame(dp_features.tolist(), columns=headers)

# Add label if present
if 'label' in df.columns:
    dp_df.insert(0, 'label', df['label'])

# Save to CSV
dp_df.to_csv('distance_pair_features_dataset.csv', index=False)
files.download('distance_pair_features_dataset.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# PortT5

In [ ]:
!pip install transformers torch sentencepiece

import pandas as pd
import torch
from transformers import T5Tokenizer, T5EncoderModel
from tqdm import tqdm

# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False)
model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")
model = model.eval()

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load your dataset
df = pd.read_csv("/content/Rna_binding_dataset.csv")

# Validate sequence column
if "sequence" not in df.columns:
    raise ValueError("Expected column named 'sequence' in the dataset.")

# Function to embed sequence with ProtT5
def embed_sequence_prott5(seq):
    seq = " ".join(list(seq))  # add space between residues
    ids = tokenizer(seq, return_tensors="pt", padding=True)
    input_ids = ids['input_ids'].to(device)
    attention_mask = ids['attention_mask'].to(device)

    with torch.no_grad():
        embedding = model(input_ids=input_ids, attention_mask=attention_mask)

    # Take mean of encoder output (excluding padding)
    last_hidden = embedding.last_hidden_state.squeeze(0)
    mask = attention_mask.squeeze(0).unsqueeze(-1)
    mean_embedding = torch.sum(last_hidden * mask, dim=0) / torch.sum(mask)
    return mean_embedding.cpu().numpy()

# Apply embedding
features = []
for seq in tqdm(df['sequence'], desc="Embedding with ProtT5"):
    emb = embed_sequence_prott5(seq)
    features.append(emb)

# Create DataFrame
embedding_dim = len(features[0])
feature_cols = [f'ProtT5_{i+1}' for i in range(embedding_dim)]
prott5_df = pd.DataFrame(features, columns=feature_cols)

# Add label if available
if 'label' in df.columns:
    prott5_df['label'] = df['label']



# Save and download
prott5_df.to_csv("prott5_features_dataset.csv", index=False)

from google.colab import files
files.download("prott5_features_dataset.csv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

Embedding with ProtT5: 100%|██████████| 17457/17457 [1:53:17<00:00,  2.57it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# BioBERT

In [ ]:
!pip install transformers sentencepiece biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.7 MB/s eta 0:00:00


In [ ]:
# Step 1: Install necessary libraries
!pip install transformers sentencepiece

# Step 2: Import libraries
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd
from Bio import SeqIO
from tqdm import tqdm

# Step 3: Load your FASTA sequences with labels
def load_fasta_with_label(filepath, label):
    records = []
    for record in SeqIO.parse(filepath, "fasta"):
        records.append({
            "sequence": str(record.seq),
            "label": label
        })
    return records

df = pd.read_csv("/content/Rna_binding_dataset.csv")

# Step 4: Load BioBERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Step 5: Feature extraction function
def extract_biobert_embeddings(sequences, tokenizer, model, max_len=512):
    embeddings = []
    for seq in tqdm(sequences):
        tokens = tokenizer(seq,
                           return_tensors='pt',
                           truncation=True,
                           padding='max_length',
                           max_length=max_len)
        input_ids = tokens['input_ids'].to(device)
        attention_mask = tokens['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden_state = outputs.last_hidden_state.squeeze(0)  # shape: (seq_len, hidden_size)
            seq_embedding = last_hidden_state.mean(dim=0).cpu().numpy()  # Mean pooling
        embeddings.append(seq_embedding)
    return np.array(embeddings)

# Step 6: Apply to your sequences
biobert_embeddings = extract_biobert_embeddings(df['sequence'], tokenizer, model)

# Step 7: Save as CSV
biobert_df = pd.DataFrame(biobert_embeddings)
biobert_df['label'] = df['label'].values
biobert_df.to_csv("/content/BioBERT_embeddings.csv", index=False)

# Step 8: Download CSV
from google.colab import files
files.download("/content/BioBERT_embeddings.csv")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

  0%|          | 1/17457 [00:02<9:53:31,  2.04s/it]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

  2%|▏         | 359/17457 [12:01<9:27:08,  1.99s/it]

# ESM2

In [ ]:
!pip install fair-esm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 5.9 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
from esm import pretrained
from tqdm import tqdm

# Load ESM-2 model (esm2_t6_8M is smallest and fastest; can upgrade later)
model, alphabet = pretrained.load_model_and_alphabet("esm2_t6_8M_UR50D")
batch_converter = alphabet.get_batch_converter()
model.eval().cuda() if torch.cuda.is_available() else model.eval()

# Load your dataset
df = pd.read_csv("/content/Rna_binding_dataset.csv")

# Check for sequence column
if 'sequence' not in df.columns:
    raise ValueError("Expected column named 'sequence' in the dataset.")

# Prepare sequences for embedding
data = [("seq{}".format(i), seq) for i, seq in enumerate(df['sequence'])]

# Extract embeddings
all_embeddings = []

with torch.no_grad():
    for i in tqdm(range(len(data)), desc="Embedding with ESM-2"):
        batch_labels, batch_strs, batch_tokens = batch_converter([data[i]])
        batch_tokens = batch_tokens.cuda() if torch.cuda.is_available() else batch_tokens
        results = model(batch_tokens, repr_layers=[6])  # Layer 6 for t6_8M model

        # Mean embedding over tokens (excluding special tokens)
        token_embeddings = results["representations"][6]
        mean_embedding = token_embeddings[0, 1:len(data[i][1]) + 1].mean(0).cpu().numpy()
        all_embeddings.append(mean_embedding)

# Convert to DataFrame
embedding_dim = len(all_embeddings[0])
esm2_cols = [f'ESM2_{i+1}' for i in range(embedding_dim)]
esm2_df = pd.DataFrame(all_embeddings, columns=esm2_cols)

# Add label column if present
if 'label' in df.columns:
    esm2_df['label'] = df['label']

# Save and download the file
esm2_df.to_csv("esm2_features_dataset.csv", index=False)

from google.colab import files
files.download("esm2_features_dataset.csv")


Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt
Embedding with ESM-2: 100%|██████████| 17457/17457 [04:11<00:00, 69.37it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# PortBERT

In [ ]:
!pip install transformers torch biopython

import pandas as pd
import torch
from transformers import BertModel, BertTokenizer
from tqdm import tqdm

# Load ProtBERT from Hugging Face
tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert")
model = model.eval()

# GPU support if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load your dataset
df = pd.read_csv("/content/Rna_binding_dataset.csv")

# Check for 'sequence' column
if 'sequence' not in df.columns:
    raise ValueError("Expected column named 'sequence'")

# Function to convert a sequence to ProtBERT embedding
def embed_sequence(sequence):
    # Formatting for ProtBERT
    sequence = " ".join(list(sequence))
    encoded_input = tokenizer(sequence, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        output = model(**{k: v.to(device) for k, v in encoded_input.items()})
    embedding = output.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding

# Generate embeddings
embeddings = []
for seq in tqdm(df['sequence'], desc="Encoding sequences with ProtBERT"):
    emb = embed_sequence(seq)
    embeddings.append(emb)

# Create DataFrame from embeddings
embedding_dim = embeddings[0].shape[0]
bert_columns = [f'ProtBERT_{i+1}' for i in range(embedding_dim)]
bert_df = pd.DataFrame(embeddings, columns=bert_columns)

# Add labels if present
if 'label' in df.columns:
    bert_df['label'] = df['label']

# Save to CSV
bert_df.to_csv("protbert_embeddings.csv", index=False)

# Let user download
from google.colab import files
files.download("protbert_embeddings.csv")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Encoding sequences with ProtBERT:  84%|████████▍ | 14661/17457 [28:34<04:57,  9.41it/s]